In [1]:
from IPython import get_ipython
#get_ipython().ast_node_interactivity = "all"
get_ipython().ast_node_interactivity = "last_expr"

## Conducción térmica:
$$\begin{array}{rll}\frac{\partial u}{\partial t} - \nabla\cdot (\kappa \nabla u) &= 0 & \text{en $\Omega\times (0,T)$,}\\
u(x,y,0) &= u_0 & \text{en }\Omega \\
u & = u_0 & \text{sobre }\Gamma_1\times (0,T) \\
\kappa \frac{\partial u}{\partial n} + \alpha(u-u_e) & = 0 & \text{sobre $\Gamma_2\times (0,T)$}\end{array}$$
con $\Omega=(0,L)\times (0,1)$, $\Gamma_1 = \{0,L\}\times(0,1)$, $\Gamma_2 = (0,L)\times \{0,1\}$, $L=6$

$u_e=25$, $\alpha=0.25$, $T=5$, $u_0(x,y)=10+\frac{90}{L}x$, y 
$\kappa(x,y) = \begin{cases} 2. & \text{ si }y<0.5\\ 0.2 & \text{ si }y>=0.5 \end{cases}$

$\Gamma_1$ tiene etiqueas asociadas 3,5; $\Gamma_2$ tiene etiquetas asociadas 2,4.

### Importamos módulos

In [2]:
import mfem.ser as mfem
import numpy as np
import time # necesario para la visualización continua

### Lectura de malla

In [3]:
mesh = mfem.Mesh('mallas/termico.mesh',1 ,1)

#### Datos del problema

In [4]:
alpha_v = 0.25
alpha_coeff = mfem.ConstantCoefficient(alpha_v)
ue_v = 25.
ue_coeff = mfem.ConstantCoefficient(ue_v*alpha_v)

class u0Coeff(mfem.PyCoefficient):
    def EvalValue(self,x):
        return 10.+90.*x[0]/6.
u0 = u0Coeff()    

# Coeficiente de la forma bilineal (definido en todo el dominio)
kappa = mfem.PWConstCoefficient(mfem.Vector([0.2,2]))

#### Etiquetas frontera

In [5]:
# para las condiciones Robin
ess_robin = mfem.intArray([1,0,1,0])

# para la condiciones Dirichlet
ess_dirich = mfem.intArray([0,1,0,1])

### Espacio de elementos finitos

In [6]:
fec = mfem.H1_FECollection(1,  mesh.Dimension())
fespace = mfem.FiniteElementSpace(mesh, fec)
print('Número de incógnitas: ' +  str(fespace.GetTrueVSize()))

Número de incógnitas: 186


#### Condición frontera

In [7]:
# arrays para definir todas las etiquetas frontera
ess_tdof_list = mfem.intArray()

# recopilamos etiquetas Dirichlet para pasarlas al solver
fespace.GetEssentialTrueDofs(ess_dirich, ess_tdof_list)
# GridFunction para valores frontera Dirichlet
x = mfem.GridFunction(fespace)
x.ProjectBdrCoefficient(u0,ess_dirich)  

### Método para ecuaciones dependientes del tiempo
Para resolver ecuaciones del tipo:
$$ \frac{\partial u}{\partial t} + {\cal L}(u)= f$$
cuya formulación sería $$\int_\Omega \frac{\partial u}{\partial t} w + \int_\Omega {\cal L}(u) w = \int_\Omega fw$$
que da lugar a 
$$ M \frac{\partial u}{\partial t} = - Ku + b$$
que se puede resolver con un método explícito: 
$$\frac{\partial u}{\partial t} = M^{-1} (-Ku + b),$$ 
o con un método implícito: 
$$M\frac{\partial u}{\partial t} = -K(u + \delta t \frac{\partial u}{\partial t} )+b $$
que da lugar a $$ (M +
\delta t K) \frac{\partial u}{\partial t} = -Ku + b \Rightarrow  \frac{\partial u}{\partial t} = (M +
\delta t K)^{-1}(-Ku + b)$$

### Formulación variacional
En nuestro caso
$$ \int_\Omega \left(\frac{\partial u}{\partial t} w + \kappa \nabla u \nabla w\right) + \int_\Gamma \alpha(u-u_e)w = 0$$
$$ \int_\Omega \frac{\partial u}{\partial t} w +  \int_\Omega \kappa \nabla u \nabla w + \int_\Gamma \alpha u w = \int_\Gamma u_ew $$
luego 
$$M \longrightarrow \int_\Omega \frac{\partial u}{\partial t} w$$
$$K \longrightarrow \int_\Omega \kappa \nabla u \nabla w + \int_\Gamma \alpha u w $$
 $$b \longrightarrow \int_\Gamma u_ew $$


### Formulación variacional

In [8]:
# Forma bilineal para M (derivada temporal)
M = mfem.BilinearForm(fespace)
M.AddDomainIntegrator(mfem.MassIntegrator())
M.Assemble()
M.Finalize()
MMat = M.SpMat()

# forma bilineal
a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.DiffusionIntegrator(kappa))
a.AddBoundaryIntegrator(mfem.MassIntegrator(alpha_coeff),ess_robin)
a.Assemble()
aMat = mfem.SparseMatrix()

# Segundo miembro (f=1)
b = mfem.LinearForm(fespace)
b.AddBoundaryIntegrator(mfem.BoundaryLFIntegrator(ue_coeff),ess_robin)
b.Assemble()
B = mfem.Vector()
X = mfem.Vector()

a.FormLinearSystem(ess_tdof_list,x,b,aMat,X,B)

### Clase para el Operador en tiempo

In [10]:
class CalorOperator(mfem.PyTimeDependentOperator):
    def __init__(self, M, K, b):
        mfem.PyTimeDependentOperator.__init__(self, M.Size())
        rel_tol = 1e-8
        self.Mmat = M
        self.Kmat = K
        self.b = b
        self.T = None
        self.current_dt = -1.0

        self.z = mfem.Vector(M.Size())
        self.zp = np.zeros(M.Size())
        # Solver para M
        self.M_prec = mfem.DSmoother()
        self.M_solver = mfem.CGSolver()
        self.M_solver.SetPreconditioner(self.M_prec)
        self.M_solver.SetOperator(M)
        self.M_solver.iterative_mode = False
        self.M_solver.SetRelTol(1e-9)
        self.M_solver.SetAbsTol(0.0)
        self.M_solver.SetMaxIter(100)
        self.M_solver.SetPrintLevel(0)

        # Solver para T = M+dtK
        self.T_prec = mfem.DSmoother()
        self.T_solver = mfem.CGSolver()
        self.T_solver.iterative_mode = False
        self.T_solver.SetRelTol(rel_tol)
        self.T_solver.SetAbsTol(0.0)
        self.T_solver.SetMaxIter(100)
        self.T_solver.SetPrintLevel(0)
        self.T_solver.SetPreconditioner(self.T_prec)        

    def Mult(self, u, u_dt):
        # Compute:
        #  du_dt = M^{-1}*(-K(u)+b) for du_dt
        self.Kmat.Mult(u, self.z)
        self.z.Neg()
        self.z += self.b
        self.M_solver.Mult(self.z, u_dt)

    def ImplicitSolve(self, dt, u, du_dt):
        # Solve the equation:
        #    du_dt = M^{-1}*[-K(u + dt*du_dt) +b]
        #    for du_dt
        if self.T is None or dt != self.current_dt:
            self.T = mfem.Add(1.0, self.Mmat, dt, self.Kmat)
            self.T_solver.SetOperator(self.T)
            self.current_dt = dt
        
        self.Kmat.Mult(u, self.z)
        self.z.Neg()
        self.z += self.b
        self.T_solver.Mult(self.z, du_dt)

#### Preparamos la salida

In [11]:
# Instante inicial
x.ProjectCoefficient(u0)

# Visualización
sout = mfem.socketstream("localhost", 19916)
sout.precision(8) 
sout << "solution\n" << mesh << x 
sout << "view 0 0\n"
sout << "pause\n"

### Resolución
#Creamos la clase, definimos el solver y hacemos iteraciones con el métod `Step`    
adv = CalorOperator(MMat, aMat, B)
ode_solver = mfem.BackwardEulerSolver()

ode_solver.Init(adv)

dt = 0.1
T = 5.
t = 0.

while t<T:
    t,dt = ode_solver.Step(x, t, dt)
    sout << "solution\n" << mesh << x 
    cad = 'Time: {0:1.2f}'.format(t)
    sout << "plot_caption '" << cad << "'"

#### Solvers:

In [ ]:
#ode_solver = mfem.ForwardEulerSolver()
#ode_solver = mfem.ESDIRK32Solver() # Runge-Kutta implícito orden 2
#ode_solver = mfem.ESDIRK33Solver()  # Runge-Kutta implícito orden 3
#ode_solver = mfem.SDIRK23Solver() # Singly-Diagonally implitic Runge-Kutta orden 2
